# AttritionIQ – EDA and Model Training
**IBM SkillsBuild · Data Analytics with AI Virtual Internship Project**

This notebook covers:
1. Dataset overview
2. Exploratory Data Analysis (EDA)
3. Feature engineering
4. ML model training and comparison
5. Model evaluation


In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.figsize': (10, 5), 'axes.spines.top': False,
                     'axes.spines.right': False, 'font.size': 11})

from data_preprocessing import load_dataset, dataset_overview, compute_attrition_stats
print('Imports OK')

## 1. Load Dataset

In [ ]:
df = load_dataset()
print(f'Shape: {df.shape}')
df.head()

In [ ]:
overview = dataset_overview(df)
print('--- Shape ---')
print(f"Rows: {overview['shape']['rows']}  |  Columns: {overview['shape']['columns']}")
print(f"\n--- Target distribution ---")
print(overview['target_distribution'])
print(f"\nAttrition rate: {overview['target_rate']}%")
print(f"\n--- Missing values ---")
mv = {k:v for k,v in overview['missing_values'].items() if v > 0}
print(mv if mv else 'None')
print(f"\nDuplicate rows: {overview['duplicate_rows']}")

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 2. Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['Attrition'].value_counts()
colors = ['#3b82d4', '#dc2626']
axes[0].bar(counts.index, counts.values, color=colors, width=0.5)
axes[0].set_title('Attrition Count')
axes[0].set_ylabel('Number of Employees')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, pctdistance=0.8)
axes[1].set_title('Attrition Distribution')

plt.tight_layout()
plt.savefig('../static/eda_target_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Attrition by Key Factors

In [ ]:
def attrition_rate_by(col, df):
    return (df.groupby(col)['Attrition'].apply(lambda x: (x=='Yes').mean()*100)
              .reset_index(name='AttritionRate'))

# Overtime
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ot = attrition_rate_by('OverTime', df)
axes[0].bar(ot['OverTime'], ot['AttritionRate'],
            color=['#3b82d4','#dc2626'], width=0.5)
axes[0].set_title('Attrition Rate by Overtime')
axes[0].set_ylabel('Attrition Rate (%)')
for i, row in ot.iterrows():
    axes[0].text(i, row.AttritionRate + 0.5, f"{row.AttritionRate:.1f}%", ha='center')

# Business Travel
bt = attrition_rate_by('BusinessTravel', df).sort_values('AttritionRate', ascending=False)
axes[1].bar(bt['BusinessTravel'], bt['AttritionRate'], color='#7c5cd8', width=0.5)
axes[1].set_title('Attrition Rate by Business Travel')
axes[1].set_ylabel('Attrition Rate (%)')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

In [ ]:
# Department and Job Role
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dept = attrition_rate_by('Department', df).sort_values('AttritionRate')
axes[0].barh(dept['Department'], dept['AttritionRate'], color='#3b82d4')
axes[0].set_title('Attrition Rate by Department')
axes[0].set_xlabel('Attrition Rate (%)')

role = attrition_rate_by('JobRole', df).sort_values('AttritionRate')
axes[1].barh(role['JobRole'], role['AttritionRate'], color='#dc2626')
axes[1].set_title('Attrition Rate by Job Role')
axes[1].set_xlabel('Attrition Rate (%)')

plt.tight_layout()
plt.show()

In [ ]:
# Satisfaction scores
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sat_cols = ['JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance']
titles   = ['Job Satisfaction', 'Environment Satisfaction', 'Work-Life Balance']
for ax, col, title in zip(axes, sat_cols, titles):
    data = attrition_rate_by(col, df)
    ax.bar(data[col].astype(str), data['AttritionRate'], color='#3b82d4', width=0.6)
    ax.set_title(f'Attrition Rate by\n{title}')
    ax.set_xlabel(title + ' (1=Low, 4=High)')
    ax.set_ylabel('Attrition Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Age and Income distributions
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for label, grp in df.groupby('Attrition'):
    axes[0].hist(grp['Age'], bins=20, alpha=0.6,
                 label=label, color='#dc2626' if label=='Yes' else '#3b82d4')
axes[0].set_title('Age Distribution by Attrition')
axes[0].set_xlabel('Age')
axes[0].legend(title='Attrition')

for label, grp in df.groupby('Attrition'):
    axes[1].hist(grp['MonthlyIncome'], bins=25, alpha=0.6,
                 label=label, color='#dc2626' if label=='Yes' else '#3b82d4')
axes[1].set_title('Monthly Income by Attrition')
axes[1].set_xlabel('Monthly Income ($)')
axes[1].legend(title='Attrition')

plt.tight_layout()
plt.show()

In [ ]:
# Years at company boxplot
groups = [df[df['Attrition']==v]['YearsAtCompany'] for v in ['No','Yes']]
plt.figure(figsize=(7, 4))
plt.boxplot(groups, labels=['Retained','Left'], patch_artist=True,
            boxprops=dict(facecolor='#dbeafe'))
plt.title('Years at Company by Attrition')
plt.ylabel('Years at Company')
plt.show()

## 4. Correlation Heatmap (Numeric Features)

In [ ]:
df_num = df.copy()
df_num['AttritionBin'] = (df['Attrition'] == 'Yes').astype(int)
num_cols = df_num.select_dtypes(include='number').columns.tolist()
# Remove constants
num_cols = [c for c in num_cols if df_num[c].nunique() > 1]
corr = df_num[num_cols].corr()

fig, ax = plt.subplots(figsize=(14, 11))
im = ax.imshow(corr, cmap='RdBu', vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols)))
ax.set_yticks(range(len(num_cols)))
ax.set_xticklabels(num_cols, rotation=90, fontsize=8)
ax.set_yticklabels(num_cols, fontsize=8)
plt.colorbar(im, ax=ax, shrink=0.7)
ax.set_title('Correlation Heatmap (Numeric Features)', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Model Training and Comparison

In [ ]:
from train_model import train_and_select
metadata = train_and_select(verbose=True)

In [ ]:
import json
print(f"Best model: {metadata['best_model']}")
print("\nModel comparison:")
rows = []
for name, m in metadata['metrics'].items():
    rows.append({'Model': name, 'Accuracy': m['accuracy'],
                 'Precision': m['precision'], 'Recall': m['recall'],
                 'F1': m['f1'], 'ROC-AUC': m['roc_auc']})
pd.DataFrame(rows).sort_values('F1', ascending=False)

In [ ]:
# Feature importance bar chart
top = metadata['top_features'][:15]
names = [f['feature'] for f in top]
vals  = [f['importance'] for f in top]

plt.figure(figsize=(10, 6))
plt.barh(names[::-1], vals[::-1], color='#3b82d4')
plt.xlabel('Feature Importance')
plt.title(f'Top 15 Features – {metadata["best_model"]}')
plt.tight_layout()
plt.show()

## 6. Model Evaluation – Best Model

In [ ]:
from evaluate_model import get_roc_curve_data, get_confusion_matrix_data
roc = get_roc_curve_data()
cm  = get_confusion_matrix_data()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC Curve
axes[0].plot([0,1],[0,1],'--',color='#ccc')
axes[0].plot(roc['fpr'], roc['tpr'], color='#dc2626', lw=2,
             label=f"AUC = {roc['auc']:.4f}")
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()

# Confusion matrix
matrix = np.array(cm['matrix'])
im = axes[1].imshow(matrix, cmap='Blues')
axes[1].set_xticks([0,1])
axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['Predicted No','Predicted Yes'])
axes[1].set_yticklabels(['Actual No','Actual Yes'])
axes[1].set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(matrix[i,j]), ha='center', va='center',
                     color='white' if matrix[i,j] > matrix.max()/2 else 'black',
                     fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
print('=== Test Set Metrics ===')
best_metrics = metadata['metrics'][metadata['best_model']]
for k, v in best_metrics.items():
    if k != 'confusion_matrix':
        print(f'  {k.capitalize():<12}: {v:.4f}')

## 7. Example Prediction

Demonstrates the end-to-end prediction pipeline.

In [ ]:
from prediction import predict_employee

high_risk = {
    'Age': 28,
    'Department': 'Sales',
    'JobRole': 'Sales Representative',
    'JobLevel': 1,
    'OverTime': 'Yes',
    'JobSatisfaction': 1,
    'WorkLifeBalance': 1,
    'MonthlyIncome': 2500,
    'YearsAtCompany': 1,
    'TotalWorkingYears': 3,
    'StockOptionLevel': 0,
    'DistanceFromHome': 25,
    'BusinessTravel': 'Travel_Frequently',
    'EnvironmentSatisfaction': 1,
    'NumCompaniesWorked': 5,
    'YearsSinceLastPromotion': 0,
    'YearsInCurrentRole': 0,
    'YearsWithCurrManager': 0,
    'Education': 2,
    'EducationField': 'Marketing',
    'RelationshipSatisfaction': 1,
    'JobInvolvement': 2,
    'PercentSalaryHike': 11,
    'PerformanceRating': 3,
    'TrainingTimesLastYear': 1,
}

result = predict_employee(high_risk)
print(f"Prediction : {result['prediction']}")
print(f"Probability: {result['probability']*100:.1f}%")
print(f"Risk Level : {result['risk_level']}")
print("\nKey Factors:")
for f in result['top_factors']:
    print(f'  - {f}')
print("\nRecommendations:")
for r in result['recommendations']:
    print(f'  • {r[:80]}...')